# Agent的高级用法-流式输出
## 1.values输出模式
当 stream_mode 设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都
要获取完整状态、状态持久化场景。

In [ ]:
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint


@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15, "total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)
# for chunk in customer_service_agent.stream(
#     {
#         "messages":[
#             {"role":"user","content":"查询客户id为cust1234的完整信息、历史订单和可用优惠"}
#         ]
#     },
#     stream_mode="values"
# ):
#     rprint(chunk)
#     print("-"*50)

## 2.updates输出模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控
Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages":[
            {"role":"user","content":"查询客户id为cust1234的完整信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-"*50)

## 3.messages输出模式（不要跑 会消失的）
该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似
ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages":[
            {"role":"user","content":"查询客户id为cust1234的完整信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    rprint(chunk)
    print("-"*50)

## 4.tasks输出模式

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages":[
            {"role":"user","content":"查询客户id为cust1234的完整信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="tasks"
):
    rprint(chunk)
    print("-"*50)

## 5.debug输出模式
该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态
持久化、工作流恢复或分布式执行跟踪的高级场景。

In [ ]:
for chunk in customer_service_agent.stream(
    {
        "messages":[
            {"role":"user","content":"查询客户id为cust1234的完整信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="debug"
):
    rprint(chunk)
    print("-"*50)

## 6.checkpoints模式

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
# 其他工具代码同上，保持不变
# ... ...

# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history,
           get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0
# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user", "content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
    config=config,
    stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

## 7.custom输出模式
开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进
度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [ ]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time


@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生成销售报告"})
    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告", "message": f"生成销售报告进度百分比：{i * 25}%"})
    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")
    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
    {"messages": [{"role": "user", "content": "生成销售报告和库存报告"}]},
    stream_mode="custom"
):
    print(chunk)
    print("-" * 50)